## Convert the .svo2 -> .mp4 + depth image file

In [1]:
import pyzed.sl as sl
import cv2
import numpy as np
import os

# 1. Setup Input/Output
input_path = 'test.svo2'
output_video_name = 'color_output.mp4'
depth_folder = 'depth_frames'
os.makedirs(depth_folder, exist_ok=True)

# 2. Initialize ZED
zed = sl.Camera()
init_params = sl.InitParameters()
init_params.set_from_svo_file(input_path)
init_params.depth_mode = sl.DEPTH_MODE.ULTRA 
init_params.coordinate_units = sl.UNIT.METER

if zed.open(init_params) != sl.ERROR_CODE.SUCCESS:
    print("Failed to open SVO file.")
    exit()

# Get SVO Information for the video writer
info = zed.get_camera_information()
width = info.camera_configuration.resolution.width
height = info.camera_configuration.resolution.height
fps = zed.get_svo_number_of_frames() / (zed.get_svo_number_of_frames() / 30) # Approximate FPS

# 3. Setup Video Writer (MP4)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_video_name, fourcc, 30, (width, height))

# Prepare Mats
image_zed = sl.Mat()
depth_zed = sl.Mat()
runtime = sl.RuntimeParameters()

print("Converting... This may take a moment.")

frame_count = 0
while True:
    if zed.grab(runtime) == sl.ERROR_CODE.SUCCESS:
        # Retrieve Color and Depth
        zed.retrieve_image(image_zed, sl.VIEW.LEFT)
        zed.retrieve_measure(depth_zed, sl.MEASURE.DEPTH)

        # --- Handle Color Video ---
        # Convert RGBA to BGR for OpenCV
        color_frame = image_zed.get_data()[:, :, :3] 
        video_writer.write(color_frame)

        # --- Handle Depth Files ---
        # Save raw float32 data (Meters) for training
        depth_data = depth_zed.get_data()
        np.save(f"{depth_folder}/depth_{frame_count:05d}.npy", depth_data)

        frame_count += 1
        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames...", end="\r")

    elif zed.grab() == sl.ERROR_CODE.END_OF_SVOFILE_REACHED:
        print("\nConversion Complete!")
        break

# Cleanup
video_writer.release()
zed.close()

[2026-03-04 11:42:38 UTC][ZED][INFO] Logging level INFO
[2026-03-04 11:42:38 UTC][ZED][INFO] Logging level INFO
[2026-03-04 11:42:38 UTC][ZED][INFO] Logging level INFO
[2026-03-04 11:42:39 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2026-03-04 11:42:39 UTC][ZED][INFO] [Init]  Serial Number: S/N 39315162
Converting... This may take a moment.
Processed 150 frames...
Conversion Complete![2026-03-04 11:43:12 UTC][ZED][WARNING] END OF SVO FILE REACHED in sl::ERROR_CODE sl::Camera::grab(sl::RuntimeParameters)

[2026-03-04 11:43:12 UTC][ZED][WARNING] END OF SVO FILE REACHED in sl::ERROR_CODE sl::Camera::grab(sl::RuntimeParameters)
